# PCA + Mejores Modelos (SVM y ANN) en Breast Cancer Wisconsin

## Objetivo
Usar las mejores configuraciones encontradas para SVM y ANN, aplicar Análisis de Componentes Principales (PCA) al dataset, y comparar el desempeño con y sin PCA. Generar conclusiones para este dataset en particular.

## Plan
- Recuperar las mejores configuraciones de SVM y ANN (mismas particiones y estandarización)
- Probar PCA con distintos `n_components`: [2, 3, 5, 10, 15, 20, 25, 30]
- Evaluar con validación cruzada (5-fold estratificado) y conjunto de prueba
- Visualizar desempeño vs número de componentes
- Conclusiones


## 1. Importación de Librerías y Configuración

**¿Qué hace esta sección?**

Importamos las bibliotecas necesarias para este análisis comparativo con PCA:

- **Todo lo de los notebooks anteriores**: sklearn, numpy, pandas, visualización
- **Nuevo: PCA (Principal Component Analysis)**:
  - `from sklearn.decomposition import PCA`
  - Técnica de reducción de dimensionalidad
  - Transforma 30 características en un número menor de "componentes principales"

**¿Qué es PCA?**

PCA es una técnica que:
1. Encuentra las direcciones de máxima varianza en los datos
2. Proyecta los datos en esas direcciones (componentes principales)
3. Permite reducir 30 dimensiones a, por ejemplo, 10, manteniendo la mayor información posible

**¿Por qué usar PCA?**

- Reduce complejidad computacional
- Elimina redundancia entre características correlacionadas
- Puede mejorar generalización al reducir ruido
- Facilita visualización (2D o 3D)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print('Librerías cargadas')


## 2. Carga, División y Estandarización de Datos

**¿Qué hace esta sección?**

Preparamos los datos EXACTAMENTE igual que en los notebooks anteriores:

1. Cargamos el mismo dataset de Breast Cancer Wisconsin
2. División train/test (80/20) con la MISMA semilla (random_state=42)
3. Estandarización con StandardScaler

**¿Por qué es importante mantenerlo igual?**

Para hacer una comparación justa entre:
- SVM sin PCA (notebook 1)
- ANN sin PCA (notebook 2)  
- SVM y ANN CON PCA (este notebook)

Usamos las MISMAS particiones de train/test para que las diferencias en resultados sean SOLO por el uso de PCA, no por diferencias en los datos.

**Nota crítica**: La estandarización se hace ANTES de PCA. PCA requiere que las características estén en la misma escala para funcionar correctamente


In [ ]:
# Cargar dataset y preparar particiones
breast_cancer = datasets.load_breast_cancer()
X = breast_cancer.data
y = breast_cancer.target

# División estratificada y estandarización
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train: {X_train_scaled.shape}, X_test: {X_test_scaled.shape}")
print(f"Clases: {np.bincount(y)}")

# Configuración de CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## 3. Configuraciones Óptimas de SVM y ANN (tomadas de notebooks previos)


In [ ]:
# Seleccionar mejores configuraciones (búsqueda rápida)
from sklearn.model_selection import GridSearchCV

# SVM: pequeña búsqueda
svm_param_grid = [
    {'kernel': ['rbf'], 'C': [1, 10, 100], 'gamma': ['scale', 0.01, 0.1]},
    {'kernel': ['linear'], 'C': [0.1, 1, 10]},
    {'kernel': ['poly'], 'degree': [2, 3], 'C': [1, 10], 'gamma': ['scale']}
]
svm_base = SVC(probability=True, random_state=42)
svm_grid = GridSearchCV(svm_base, svm_param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0)
svm_grid.fit(X_train_scaled, y_train)
svm_best = svm_grid.best_estimator_
print('Mejor SVM:', svm_grid.best_params_, 'CV:', round(svm_grid.best_score_, 4))

# ANN: pequeña búsqueda
mlp_param_grid = {
    'hidden_layer_sizes': [(20, 10), (50, 30, 10), (100,)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'lbfgs'],
    'alpha': [0.0001, 0.001]
}
mlp_base = MLPClassifier(max_iter=2000, random_state=42, early_stopping=True, validation_fraction=0.1)
mlp_grid = GridSearchCV(mlp_base, mlp_param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0)
mlp_grid.fit(X_train_scaled, y_train)
mlp_best = mlp_grid.best_estimator_
print('Mejor ANN:', mlp_grid.best_params_, 'CV:', round(mlp_grid.best_score_, 4))


## 4. Aplicar PCA y Evaluar con CV y Test


In [ ]:
n_components_list = [2, 3, 5, 10, 15, 20, 25, 30]

results_pca = []

# Rendimiento sin PCA (baseline)
def evaluate_model(model, X_tr, y_tr, X_te, y_te, name):
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='accuracy')
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_te)[:, 1]
    else:
        # Para SVM sin probability, usar decision_function
        proba = model.decision_function(X_te)
    return {
        'model': name,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'test_acc': accuracy_score(y_te, y_pred),
        'test_f1': f1_score(y_te, y_pred),
        'test_auc': roc_auc_score(y_te, proba)
    }

print('Evaluando baseline (sin PCA)...')
baseline_svm = evaluate_model(svm_best, X_train_scaled, y_train, X_test_scaled, y_test, 'SVM (sin PCA)')
baseline_ann = evaluate_model(mlp_best, X_train_scaled, y_train, X_test_scaled, y_test, 'ANN (sin PCA)')
print('Baseline SVM:', baseline_svm)
print('Baseline ANN:', baseline_ann)

for n in n_components_list:
    print(f"\nAplicando PCA con n_components={n} ...")
    pca = PCA(n_components=n, random_state=42)
    # PCA se aplica sobre datos estandarizados
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    # Evaluar SVM y ANN con PCA
    res_svm = evaluate_model(svm_best, X_train_pca, y_train, X_test_pca, y_test, f'SVM (PCA={n})')
    res_ann = evaluate_model(mlp_best, X_train_pca, y_train, X_test_pca, y_test, f'ANN (PCA={n})')

    # Guardar
    res_svm['n_components'] = n
    res_ann['n_components'] = n
    results_pca.extend([res_svm, res_ann])

results_df = pd.DataFrame(results_pca)
print('\nResultados PCA (head):')
print(results_df.head())


## 5. Visualización de Resultados vs Componentes (PCA)

**¿Qué hace esta sección?**

Crea gráficas que muestran cómo cambia el desempeño según el número de componentes de PCA.

**Gráficas generadas**:

1. **CV Accuracy vs n_components**:
   - Eje X: Número de componentes (5, 10, 15, 20, 25, 30)
   - Eje Y: Accuracy en validación cruzada
   - Dos líneas: una para SVM, otra para ANN
   - **Qué buscar**: ¿En qué punto se estabiliza el desempeño?

2. **Test Accuracy vs n_components**:
   - Similar a la anterior pero con datos de prueba
   - Más importante porque muestra el desempeño en datos no vistos
   - **Qué buscar**: ¿Con cuántos componentes alcanzamos el máximo?

3. **Test AUC vs n_components**:
   - Muestra la capacidad de discriminación
   - AUC es más robusto que accuracy para datos desbalanceados
   - **Qué buscar**: ¿Se mantiene alto AUC con pocos componentes?

**Interpretación de las curvas**:

- **Curva ascendente**: Más componentes = mejor desempeño
- **Curva plana**: Alcanzó el máximo, más componentes no ayudan
- **Curva descendente**: Demasiados componentes pueden añadir ruido

**Punto óptimo**:
- El "codo" de la curva: donde se estabiliza el desempeño
- Balance entre complejidad (pocos componentes) y desempeño (alta accuracy)

**Tabla resumen**: Muestra todas las métricas para cada configuración


In [ ]:
# Curvas de desempeño por número de componentes
plt.figure(figsize=(14, 10))

for metric, idx, title in [('cv_mean', 1, 'CV Accuracy'), ('test_acc', 2, 'Test Accuracy'), ('test_auc', 3, 'Test AUC')]:
    plt.subplot(3, 1, idx)
    # SVM
    subset_svm = results_df[results_df['model'].str.startswith('SVM')]
    plt.plot(subset_svm['n_components'], subset_svm[metric], marker='o', label='SVM')
    # ANN
    subset_ann = results_df[results_df['model'].str.startswith('ANN')]
    plt.plot(subset_ann['n_components'], subset_ann[metric], marker='o', label='ANN')
    plt.title(f'{title} vs n_components (PCA)')
    plt.xlabel('n_components')
    plt.ylabel(title)
    plt.grid(True, alpha=0.3)
    plt.legend()

plt.tight_layout()
plt.show()

# Tabla resumen
summary = results_df.pivot_table(index='n_components', columns='model', values=['cv_mean','test_acc','test_auc'])
print('Resumen por número de componentes:')
print(summary.round(4))


## 6. Conclusiones para este Dataset

**¿Qué hace esta sección?**

Resume los hallazgos del análisis PCA + Modelos y proporciona conclusiones específicas para este dataset.

**Análisis presentado**:

1. **Mejor resultado con PCA por modelo**:
   - Para SVM: ¿Con cuántos componentes funciona mejor?
   - Para ANN: ¿Con cuántos componentes funciona mejor?
   - Comparación con baseline (sin PCA)

2. **Mejor resultado global**:
   - La combinación óptima de (Modelo + n_components)
   - ¿PCA mejoró o empeoró el desempeño?

3. **Comparación con baseline sin PCA**:
   - ¿Vale la pena usar PCA en este dataset?
   - ¿Ganamos algo reduciendo dimensionalidad?

**Observaciones típicas en Breast Cancer Wisconsin**:

- **Alta redundancia**: Las 30 características están correlacionadas
- **PCA efectivo**: ~10 componentes retienen >95% de varianza
- **SVM robusto**: SVM mantiene desempeño con PCA
- **ANN beneficiado**: ANN puede mejorar con PCA (menos ruido)
- **Reducción exitosa**: De 30 a 10-15 componentes sin pérdida significativa

**Conclusión práctica**:

Este dataset es ideal para PCA porque:
- Tiene características correlacionadas (radio, perímetro, área están relacionados)
- PCA elimina esta redundancia
- Simplifica el modelo sin sacrificar accuracy

**Recomendación pipeline**:
```
Data → StandardScaler → PCA(10-15 componentes) → SVM/ANN → Predicciones
```

**Ventajas**:
- Modelos más rápidos (menos características)
- Menos overfitting (menos ruido)
- Mejor generalización
- Interpretación más simple


In [ ]:
print("\n" + "="*80)
print("CONCLUSIONES PCA + MEJORES MODELOS (SVM y ANN)")
print("="*80)

# Mejor resultado por modelo y global
best_by_model = results_df.sort_values(['model','test_acc'], ascending=[True, False]).groupby('model').head(1)
print("\nMejores resultados por modelo (con PCA):")
print(best_by_model[['model','n_components','cv_mean','test_acc','test_auc']].round(4))

best_overall = results_df.loc[results_df['test_acc'].idxmax()]
print(f"\nMejor resultado global con PCA: {best_overall['model']} con n_components={best_overall['n_components']}\n"
      f"CV={best_overall['cv_mean']:.4f}, Test Acc={best_overall['test_acc']:.4f}, AUC={best_overall['test_auc']:.4f}")

# Comparar con baseline sin PCA
print("\nBaseline sin PCA:")
print(pd.DataFrame([baseline_svm, baseline_ann]).round(4))

print("\nObservaciones (típicas en este dataset):")
print("• El dataset Breast Cancer Wisconsin tiene alta redundancia entre características (correlaciones altas).")
print("• PCA suele permitir mantener >95% de varianza con ~10 componentes, reduciendo dimensionalidad.")
print("• SVM con RBF/lineal mantiene o mejora levemente el desempeño con PCA moderado (5-15 componentes).")
print("• ANN puede beneficiarse de PCA (menos ruido y variables), mejorando estabilidad y generalización.")
print("• Si n_components es muy bajo (2-3), puede perderse información y bajar el desempeño.")

print("\nConclusión práctica:")
print("• Para este dataset, PCA con ~10-15 componentes suele ofrecer un buen compromiso entre simplicidad y desempeño.")
print("• SVM se muestra robusta al PCA; ANN mejora en estabilidad con PCA moderado.")
print("• Recomendación: mantener pipeline: StandardScaler → PCA(10-15) → Modelo (SVM/ANN) y validar con CV.")

print("\n" + "="*80)


# Análisis Comparativo: SVM vs ANN con y sin PCA

## Objetivo

**¿Cuál es el propósito de este análisis?**

Este notebook realiza un análisis comparativo exhaustivo para responder:

1. **¿PCA mejora o empeora el desempeño de los modelos?**
2. **¿Cuántos componentes principales son óptimos?**
3. **¿Afecta PCA de manera diferente a SVM vs ANN?**
4. **¿Podemos reducir dimensionalidad manteniendo accuracy?**

## Dataset

- **Nombre**: Breast Cancer Wisconsin (Diagnostic) Data Set
- **Fuente**: UCI Machine Learning Repository
- **Características originales**: 30 características numéricas
- **Análisis**: Comparación sistemática con/sin reducción de dimensionalidad

**Características del dataset relevantes para PCA**:
- Muchas características están altamente correlacionadas
- Por ejemplo: radio, perímetro y área están matemáticamente relacionados
- Esto crea redundancia que PCA puede eliminar

## Metodología

**Pipeline experimental**:

1. **Aplicar PCA** con diferentes números de componentes (5, 10, 15, 20, 25, 30)
   - Permite ver la curva de desempeño vs dimensionalidad
   
2. **Usar mejores modelos** de SVM y ANN encontrados en notebooks anteriores
   - SVM: Configuración óptima (kernel, C, gamma)
   - ANN: Configuración óptima (arquitectura, activación, optimizador)
   
3. **Comparar desempeño** con y sin PCA
   - Baseline sin PCA como punto de referencia
   - Múltiples métricas: Accuracy, F1, AUC
   
4. **Analizar varianza explicada** por componentes principales
   - ¿Cuánta información retenemos con N componentes?
   - ¿Dónde está el punto de rendimientos decrecientes?
   
5. **Extraer conclusiones** específicas para este dataset
   - Recomendaciones prácticas
   - Trade-offs entre complejidad y desempeño


## 1. Importación de Librerías


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_curve, auc
import sklearn
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("Librerías importadas correctamente")
print("Versiones de librerías principales:")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")


## 2. Carga y Preparación de Datos


In [ ]:
# Cargar el dataset Breast Cancer Wisconsin
breast_cancer = datasets.load_breast_cancer()

# Separar características y etiquetas
X = breast_cancer.data
y = breast_cancer.target

print("Información del Dataset:")
print(f"Forma del dataset: {X.shape}")
print(f"Características originales: {X.shape[1]}")
print(f"Muestras: {X.shape[0]}")
print(f"Clases: {len(np.unique(y))}")
print(f"Nombres de clases: {breast_cancer.target_names}")

# División en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nDivisión de datos:")
print(f"Conjunto de entrenamiento: {X_train.shape[0]} muestras")
print(f"Conjunto de prueba: {X_test.shape[0]} muestras")

# Estandarización de características (CRÍTICO para PCA y modelos)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nDatos estandarizados:")
print(f"Media de X_train_scaled: {np.mean(X_train_scaled):.4f}")
print(f"Desviación estándar de X_train_scaled: {np.std(X_train_scaled):.4f}")

# Configurar validación cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## 3. Definición de Mejores Modelos (SVM y ANN)


In [ ]:
# Definir los mejores modelos basados en los análisis anteriores
# (Estos serían los mejores modelos encontrados en los notebooks anteriores)

# Mejor modelo SVM (basado en análisis anterior)
best_svm = SVC(
    kernel='rbf',  # Kernel RBF generalmente fue el mejor
    C=10.0,        # Parámetro C optimizado
    gamma='scale', # Gamma optimizado
    random_state=42
)

# Mejor modelo ANN (basado en análisis anterior)
best_ann = MLPClassifier(
    hidden_layer_sizes=(50, 30, 10),  # Arquitectura profunda que funcionó bien
    activation='relu',                # ReLU fue la mejor función de activación
    solver='adam',                    # Adam fue el mejor optimizador
    alpha=0.001,                      # Regularización L2
    learning_rate_init=0.001,         # Learning rate optimizado
    max_iter=2000,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20
)

print("Mejores modelos definidos:")
print("=" * 40)
print("SVM:")
print(f"  Kernel: RBF")
print(f"  C: 10.0")
print(f"  Gamma: scale")
print(f"  Random state: 42")

print("\nANN:")
print(f"  Arquitectura: (50, 30, 10)")
print(f"  Activación: ReLU")
print(f"  Optimizador: Adam")
print(f"  Alpha: 0.001")
print(f"  Learning rate: 0.001")
print(f"  Early stopping: True")


## 4. Análisis de Componentes Principales (PCA)

**¿Qué hace esta sección?**

**AQUÍ SE APLICA PCA Y SE ANALIZA LA REDUCCIÓN DE DIMENSIONALIDAD**

Esta es la sección más importante de este notebook.

**Proceso**:

1. **Aplicar PCA completo**: 
   - Primero aplicamos PCA con todos los componentes (30) para análisis
   - Esto nos permite ver cuánta varianza explica cada componente

2. **Analizar varianza explicada**:
   - **Varianza explicada**: Qué porcentaje de la información original retiene cada componente
   - Por ejemplo: Si PC1 tiene 0.45 = retiene 45% de toda la información
   - **Varianza acumulada**: Suma acumulativa
   - Por ejemplo: Con 10 componentes podemos retener 95% de la información

3. **Aplicar PCA con diferentes n_components**:
   - Probamos con 5, 10, 15, 20, 25, 30 componentes
   - Para cada valor, transformamos los datos: 30 dimensiones → n dimensiones
   - `pca.fit_transform(X_train)` aprende de train y transforma
   - `pca.transform(X_test)` solo transforma test (evita data leakage)

**Gráficas generadas**:

1. **Varianza por componente**: Cuánta información aporta cada PC individual
2. **Varianza acumulada**: Curva que muestra cuántos componentes necesitamos para X% de varianza
3. **Comparación por n_components**: Varianza total retenida con diferentes números de componentes

**Interpretación**:
- Si con 10 componentes retenemos 95% de varianza, ¡redujimos de 30 a 10 perdiendo solo 5%!
- La línea roja (95%) y verde (99%) muestran umbrales comunes


In [ ]:
# Aplicar PCA con diferentes números de componentes
n_components_list = [5, 10, 15, 20, 25, 30]  # Incluimos 30 para comparar con datos originales
pca_results = {}

print("Análisis de Componentes Principales (PCA):")
print("=" * 50)

# Aplicar PCA completo para análisis de varianza
pca_full = PCA()
pca_full.fit(X_train_scaled)

# Calcular varianza explicada acumulada
explained_variance_ratio = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance_ratio)

print(f"Varianza explicada por los primeros componentes:")
for i in range(min(10, len(explained_variance_ratio))):
    print(f"  PC{i+1:2d}: {explained_variance_ratio[i]:.4f} ({explained_variance_ratio[i]*100:.2f}%)")

print(f"\nVarianza acumulada:")
for n in n_components_list:
    if n <= len(cumulative_variance):
        print(f"  {n:2d} componentes: {cumulative_variance[n-1]:.4f} ({cumulative_variance[n-1]*100:.2f}%)")

# Aplicar PCA con diferentes números de componentes
for n_components in n_components_list:
    if n_components <= X_train_scaled.shape[1]:
        pca = PCA(n_components=n_components, random_state=42)
        X_train_pca = pca.fit_transform(X_train_scaled)
        X_test_pca = pca.transform(X_test_scaled)
        
        pca_results[n_components] = {
            'pca': pca,
            'X_train': X_train_pca,
            'X_test': X_test_pca,
            'explained_variance_ratio': pca.explained_variance_ratio_,
            'cumulative_variance': np.cumsum(pca.explained_variance_ratio_)[-1]
        }
        
        print(f"\nPCA con {n_components} componentes:")
        print(f"  Forma de datos transformados: {X_train_pca.shape}")
        print(f"  Varianza explicada total: {pca_results[n_components]['cumulative_variance']:.4f}")

# Visualizar varianza explicada
plt.figure(figsize=(15, 5))

# Gráfico 1: Varianza explicada por componente
plt.subplot(1, 3, 1)
plt.bar(range(1, 11), explained_variance_ratio[:10], alpha=0.7)
plt.xlabel('Componente Principal')
plt.ylabel('Varianza Explicada')
plt.title('Varianza Explicada por Componente')
plt.grid(True, alpha=0.3)

# Gráfico 2: Varianza acumulada
plt.subplot(1, 3, 2)
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 'bo-', alpha=0.7)
plt.axhline(y=0.95, color='r', linestyle='--', alpha=0.7, label='95% varianza')
plt.axhline(y=0.99, color='g', linestyle='--', alpha=0.7, label='99% varianza')
plt.xlabel('Número de Componentes')
plt.ylabel('Varianza Acumulada')
plt.title('Varianza Acumulada')
plt.legend()
plt.grid(True, alpha=0.3)

# Gráfico 3: Comparación de varianza para diferentes n_components
plt.subplot(1, 3, 3)
n_comp_values = list(pca_results.keys())
var_values = [pca_results[n]['cumulative_variance'] for n in n_comp_values]
plt.plot(n_comp_values, var_values, 'ro-', alpha=0.7)
plt.xlabel('Número de Componentes')
plt.ylabel('Varianza Acumulada')
plt.title('Varianza por Número de Componentes')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Evaluación de Modelos con y sin PCA

**¿Qué hace esta sección?**

**AQUÍ SE ENTRENAN Y COMPARAN LOS MODELOS CON Y SIN PCA**

Esta sección realiza el experimento principal del notebook.

**Experimento**:

1. **Baseline (sin PCA)**:
   - Entrenar SVM con las 30 características originales
   - Entrenar ANN con las 30 características originales
   - Estos son nuestros puntos de referencia

2. **Con PCA (diferentes n_components)**:
   - Para cada valor de n_components (5, 10, 15, 20, 25, 30):
     - Aplicar PCA para reducir dimensionalidad
     - Entrenar SVM con los datos reducidos
     - Entrenar ANN con los datos reducidos
     - Evaluar con validación cruzada y conjunto de prueba

**Métricas evaluadas**:
- CV Accuracy (validación cruzada)
- Test Accuracy
- Test F1-Score
- Test AUC-ROC

**Lo que buscamos descubrir**:
- ¿PCA mejora o empeora el desempeño?
- ¿Cuántos componentes son óptimos?
- ¿Afecta igual a SVM que a ANN?
- ¿Podemos reducir dimensionalidad sin perder accuracy?

**Importante**: Los modelos usados son las configuraciones óptimas encontradas en notebooks anteriores


In [ ]:
# Función para evaluar un modelo con validación cruzada
def evaluate_model(model, X_train, X_test, y_train, y_test, cv, model_name, data_type):
    """
    Evalúa un modelo y retorna métricas completas
    
    Esta función:
    1. Realiza validación cruzada para estimar desempeño
    2. ENTRENA el modelo con todos los datos de entrenamiento
    3. Hace predicciones en el conjunto de prueba
    4. Calcula todas las métricas relevantes
    
    Parámetros:
    - model: Modelo SVM o ANN a evaluar
    - X_train, X_test: Datos de entrenamiento y prueba (pueden tener PCA aplicado)
    - y_train, y_test: Etiquetas
    - cv: Configuración de validación cruzada
    - model_name: Nombre del modelo (SVM o ANN)
    - data_type: Tipo de datos (Original, PCA=5, PCA=10, etc.)
    """
    try:
        # PASO 1: Validación cruzada (estimación robusta del desempeño)
        cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
        
        # PASO 2: ENTRENAR el modelo con TODOS los datos de entrenamiento
        # Esta es la línea donde realmente se entrena el modelo
        model.fit(X_train, y_train)
        
        # PASO 3: Predecir en conjunto de prueba (datos nunca vistos)
        y_pred = model.predict(X_test)
        # Obtener probabilidades si el modelo lo permite (para AUC)
        y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
        
        # PASO 4: Calcular todas las métricas de evaluación
        accuracy = accuracy_score(y_test, y_pred)  # % correctas
        precision = precision_score(y_test, y_pred)  # Precisión en positivos
        recall = recall_score(y_test, y_pred)  # Sensibilidad
        f1 = f1_score(y_test, y_pred)  # Balance precision-recall
        auc_score = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
        
        # PASO 5: Retornar diccionario con todos los resultados
        return {
            'model_name': model_name,
            'data_type': data_type,
            'cv_mean': cv_scores.mean(),  # Promedio de CV
            'cv_std': cv_scores.std(),  # Desviación estándar de CV
            'test_accuracy': accuracy,
            'test_precision': precision,
            'test_recall': recall,
            'test_f1': f1,
            'test_auc': auc_score,
            'cv_scores': cv_scores,
            'predictions': y_pred,
            'probabilities': y_pred_proba
        }
    except Exception as e:
        print(f"Error evaluando {model_name} con {data_type}: {str(e)}")
        return None

# Evaluar modelos sin PCA (datos originales)
print("Evaluando modelos SIN PCA (datos originales):")
print("=" * 60)

# Evaluar SVM sin PCA
svm_original = evaluate_model(
    SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42, probability=True),
    X_train_scaled, X_test_scaled, y_train, y_test, cv,
    'SVM', 'Original'
)

# Evaluar ANN sin PCA
ann_original = evaluate_model(
    MLPClassifier(
        hidden_layer_sizes=(50, 30, 10),
        activation='relu',
        solver='adam',
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=2000,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20
    ),
    X_train_scaled, X_test_scaled, y_train, y_test, cv,
    'ANN', 'Original'
)

print(f"SVM - Original:")
print(f"  CV Accuracy: {svm_original['cv_mean']:.4f} (+/- {svm_original['cv_std'] * 2:.4f})")
print(f"  Test Accuracy: {svm_original['test_accuracy']:.4f}")
print(f"  Test F1-Score: {svm_original['test_f1']:.4f}")

print(f"\nANN - Original:")
print(f"  CV Accuracy: {ann_original['cv_mean']:.4f} (+/- {ann_original['cv_std'] * 2:.4f})")
print(f"  Test Accuracy: {ann_original['test_accuracy']:.4f}")
print(f"  Test F1-Score: {ann_original['test_f1']:.4f}")
